## Main Question
What are the correct RUL labels, and how do we split engines for development?

In [152]:
import pandas as pd

# loading the first training dataset
df_train = pd.read_csv("../CMAPSSData/train_FD001.txt", sep=r"\s+", header=None)

# checking if it loaded correctly
print(df_train.head(5))
print(df_train.info())

   0   1       2       3      4       5       6        7        8      9   \
0   1   1 -0.0007 -0.0004  100.0  518.67  641.82  1589.70  1400.60  14.62   
1   1   2  0.0019 -0.0003  100.0  518.67  642.15  1591.82  1403.14  14.62   
2   1   3 -0.0043  0.0003  100.0  518.67  642.35  1587.99  1404.20  14.62   
3   1   4  0.0007  0.0000  100.0  518.67  642.35  1582.79  1401.87  14.62   
4   1   5 -0.0019 -0.0002  100.0  518.67  642.37  1582.85  1406.22  14.62   

   ...      16       17       18      19    20   21    22     23     24  \
0  ...  521.66  2388.02  8138.62  8.4195  0.03  392  2388  100.0  39.06   
1  ...  522.28  2388.07  8131.49  8.4318  0.03  392  2388  100.0  39.00   
2  ...  522.42  2388.03  8133.23  8.4178  0.03  390  2388  100.0  38.95   
3  ...  522.86  2388.08  8133.83  8.3682  0.03  392  2388  100.0  38.88   
4  ...  522.19  2388.04  8133.80  8.4294  0.03  393  2388  100.0  38.90   

        25  
0  23.4190  
1  23.4236  
2  23.3442  
3  23.3739  
4  23.4044  

[5 rows

Now we add the RUL column:

In [153]:
train_engine_ids = df_train[0].unique()

for engine_id in train_engine_ids:
    cycle_ids = df_train.loc[df_train[0] == engine_id][1]
    lifetime = max(cycle_ids)
    df_train.loc[df_train[0] == engine_id , "RUL"] = lifetime - df_train[1]

df_train.to_csv("../outputs/TRAIN_FD001.csv", index=False)


Next we read the test_FD001 and RUL_FD001 files:

In [154]:
df_test = pd.read_csv("../CMAPSSData/test_FD001.txt", sep=r"\s+", header=None)
df_rul = pd.read_csv("../CMAPSSData/RUL_FD001.txt", sep=r"\s+", header=None)

print(f"Amount of engines in df_rul: {len(df_rul)}")
print(f"Amount of engines in df_test: {len(df_test[0].unique())}")



Amount of engines in df_rul: 100
Amount of engines in df_test: 100


Now we want to write the ACTUAL RUL from the df_rul into a new column in the test data df_test:

In [155]:
test_engine_ids = df_test[0].unique()

for engine_id in test_engine_ids:
    cycle_ids = df_test.loc[df_test[0] == engine_id][1]
    lifetime = max(cycle_ids)
    df_test.loc[df_test[0] == engine_id , "RUL"] = lifetime + df_rul.loc[engine_id-1][0] - df_test[1]

df_test.to_csv("../outputs/TEST_FD001.csv", index=False)

Now both the train and the test dataset have the RUL added and we need to split training into fitting and validation groups

In [156]:
from sklearn.model_selection import train_test_split

train_engine_ids = df_train[0].unique()
fitting_items, validation_items = train_test_split(
    train_engine_ids,
    test_size = 0.2,
    random_state=8
)

print(fitting_items)
print(validation_items)

fitting_mask = df_train[0].isin(fitting_items)

df_fitting = df_train.loc[fitting_mask].copy()
df_validation = df_train.loc[~fitting_mask].copy()

df_fitting.to_csv("../outputs/FITTING_FD001.csv", index=False)
df_validation.to_csv("../outputs/VALIDATION_FD001.csv", index=False)

[58 65 81 25 71 93 17 89 20 80 41 74 36 51 35 77 33 90  7 23 47 82 53 38
  5 88 78 26 37 59 55 39  3 70  8  1 11 48 66 97 29 96 22 46 21 32 94 95
 79 67 27 43 83 72 76 54 69 52 92 57 15 19 60 16 62 10 14 28 75 50 61 86
 49 64 84  9 91  6 85 68]
[ 18  24   2  45  56 100  87  30  63  34  99  13  12  40   4  42  98  31
  73  44]
